In [34]:
%load_ext autoreload
%autoreload 2

In [57]:
# REQUIRES
# ========
# System libraries
# ----------------
import time

# Third-party libraries
# ----------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# MODULES
# ========
# System modules
# --------------
from typing import Union
from pathlib import Path
from datetime import datetime

# Custom modules
# Defines
# -------
from src.config.globals import *

# Helpers functions
# -----------------
from src.helpers import funcs as hlp



In [58]:
# CHECK CHANGES ONF IMPORT FILES
# ==============================
import importlib
importlib.reload(hlp)

print(hlp.__file__)
print(hasattr(hlp, "SectionBegin"))
print([name for name in dir(hlp) if "Section" in name])

/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/helpers/funcs.py
True
['SectionBegin', 'SectionBreak', 'SectionEnd']


In [36]:
# PATHS
# =====
# CURRENT FILE PATH
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = (NOTEBOOK_DIR.parent / "data").resolve()

# Global variables definition for this notebook
# ---------------------------------------------
# dataset:str = "../data/ventas_con_problemas.csv"
dataset: str = DATA_DIR / "ventas_con_problemas.csv"

# Pandas dataframe to hold the dataset
df: pd.DataFrame

In [54]:
# LOAD SOURCE DATASET (from CSV file)
# ===================================
hlp.SectionBegin(f"DATASET PATH: \n{dataset}", 0)
hlp.SectionBreak(0, 0)

try:
  df = pd.read_csv(dataset)
  print(f"File loaded successfully. Total of original rows: {len(df)}")
  print(f"First 5 rows of the dataset:")
  print(df.head())
  
except FileNotFoundError:
  print(f"Error: Please, check path. \n'{dataset}'")
  print(f"Error: File not found.")

hlp.SectionEnd(0)


+ =========================================================
+ DATASET PATH: 
/home/dcueli/projects/courses/regulated/ai-and-big-data/p1/bigdata_sys/src/activities/RA1/lesson_02/data/ventas_con_problemas.csv
+ ---------------------------------------------------------
File loaded successfully. Total of original rows: 320
First 5 rows of the dataset:
  fecha             producto  cantidad  precio_unitario   total
0   NaN  Monitor 24 pulgadas       1.0           114.18  114.18
1   NaN      Cargador rapido       6.0            37.74  226.44
2   NaN  Monitor 24 pulgadas       5.0           193.91  969.55
3   NaN     Altavoz portatil       8.0           111.84  894.72
4   NaN      Cargador rapido       2.0            15.47   30.94
+ =========================================================


In [ ]:
# DATA CLEANING
# =============
# Audit & detection of null values in the dataset
# -----------------------------------------------
tInit: float = time.time()
hlp.SectionBegin("AUDIT & DETECTION OF NULL VALUES", 0)
hlp.SectionBreak(0,0)

# Total null values per column in the dataset
nullValuesPerColumn: int = df.isnull().sum()
print("Null values per column:")
print(nullValuesPerColumn)
hlp.SectionBreak(0,0)

# Total number of rows with at least one null value
rowsWithNulls: pd.DataFrame = df[df.isnull().any(axis=1)]
print(f"Total number of rows with at least one null value: \n{len(rowsWithNulls)}")

hlp.SectionBreak(0,0)
tEnd: float = time.time()
print(f"+ Time to load: {tEnd - tInit:.4f} seconds")
hlp.SectionEnd(0)


+ =========================================================
+ AUDIT & DETECTION OF NULL VALUES
+ ---------------------------------------------------------
Null values per column:
fecha              5
producto           5
cantidad           5
precio_unitario    5
total              5
dtype: int64
+ ---------------------------------------------------------
Total number of rows with at least one null value: 
25
+ ---------------------------------------------------------
+ Time to load: 0.0048 seconds
+ =========================================================


In [ ]:
# ToProperDate
# ------------
# Try to convert the date column to proper datetime format
# errors='coerce' converts all invalid dates to NaN
# 
# @param string pColumn Name of the column to convert to proper date format.
# @return None
def ToProperDate(pColumn:str = 'fecha') -> None:
  properDateTime: datetime | pd.Timestamp = pd.to_datetime(df[pColumn], errors='coerce')

  # How many dates were successfully converted and how many failed
  totRows: int = len(df)
  totProperDates: int = properDateTime.notna().sum()
  totInvalidDates: int = properDateTime.isna().sum()

  print(f"Total rows: {totRows}")
  print(f"Valid dates converted successfully: {totProperDates}")
  print(f"Invalid or null dates detected: {totInvalidDates}")

  # Show some of the rows with dates that failed
  print("\nExamples of rows with invalid dates:")
  print(df[properDateTime.isna()][pColumn].head(10))  
  
  
tInit: float = time.time()
hlp.SectionBegin("AUDIT & DETECTION OF INVALID DATE VALUES", 0)
print("+ Summary of conversion date column state:")
hlp.SectionBreak(0,0)
ToProperDate("fecha")
hlp.SectionBreak(0,0)

tEnd: float = time.time()
print(f"+ Time to load: {tEnd - tInit:.4f} seconds")
hlp.SectionEnd(0)


+ =========================================================
+ AUDIT & DETECTION OF INVALID DATE VALUES
+ Summary of date states:
+ ---------------------------------------------------------
Total rows: 320
Valid dates converted successfully: 302
Invalid or null dates detected: 18

Examples of rows with invalid dates:
0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
25     2026-02-30
26     2026-13-10
27     2026-10-45
28    no_es_fecha
29     31/02/2026
Name: fecha, dtype: str
+ ---------------------------------------------------------
+ Time to load: 0.0054 seconds
+ =========================================================


In [ ]:
# PROCESSING TEXT IN ESPAÑOL
# ==========================
tInit = time.time()

# 1. Convertir a numérico forzando errores a NaN por seguridad
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')
df['precio_unitario'] = pd.to_numeric(df['precio_unitario'], errors='coerce')
df['total'] = pd.to_numeric(df['total'], errors='coerce')

# 2. Identificar registros con valores <= 0
cantidades_negativas = df[df['cantidad'] <= 0]
precios_negativos = df[df['precio_unitario'] <= 0]
totales_negativos = df[df['total'] <= 0]

# 3. Mostrar resumen de los hallazgos
print(f"Filas con cantidad <= 0: {len(cantidades_negativas)}")
print(f"Filas con precio_unitario <= 0: {len(precios_negativos)}")
print(f"Filas con total <= 0: {len(totales_negativos)}")

print("\n--- Ejemplo de filas con importes o cantidades negativas ---")
print(df[(df['cantidad'] <= 0) | (df['precio_unitario'] <= 0) | (df['total'] <= 0)].head(5))


+ =========================================================
+ LANGUAGE MODEL: -Spanish-
+ Persons proper names extracted from the text (PER label):
+ ---------------------------------------------------------
+ 
Proper name: Lucas
Proper name: Rodrigor
Proper name: Rodrigor
Proper name: Caballo
Proper name: wan minim
Proper name: amet voluptate
Proper name: Labore
Proper name: Dolor
Proper name: Lucas
Proper name: Lucas
+ 
+ All detected entities (for comparison):
+ ---------------------------------------
Text: Lorem                     | Type/Label: MISC
Text: occaecat pecador          | Type/Label: MISC
Text: Aute ut aliquia diodeno   | Type/Label: MISC
Text: Lucas                     | Type/Label: PER
Text: No puedor pecador papaar  | Type/Label: MISC
Text: Rodrigor                  | Type/Label: ORG
Text: occaecat                  | Type/Label: LOC
Text: peich                     | Type/Label: LOC
Text: Irure                     | Type/Label: ORG
Text: Magna                     | Ty

In [16]:
# PROCESSING TEXT IN LATIN
# ========================
tInit = time.time()
doc_LA = nlp_LA(text)

# Process text with model
ProcessTextWithModel(doc_LA, "Latin")

#Show the text directly with interactive tags and colors
displacy.render(doc_LA, style="ent", jupyter=True)

+ =========================================================
+ LANGUAGE MODEL: -Latin-
+ Persons proper names extracted from the text (PER label):
+ ---------------------------------------------------------
+ 
Proper name: Lucas
Proper name: Rodrigor
Proper name: Magna
Proper name: Rodrigor
Proper name: Rodrigor
Proper name: Caballo
Proper name: Dolor
Proper name: Hasta
Proper name: Lucas
Proper name: Lucas
+ 
+ All detected entities (for comparison):
+ ---------------------------------------
Text: Lucas                     | Type/Label: PERSON
Text: Rodrigor                  | Type/Label: PERSON
Text: La                        | Type/Label: NORP
Text: Irure                     | Type/Label: LOC
Text: Magna                     | Type/Label: PERSON
Text: Rodrigor                  | Type/Label: PERSON
Text: Rodrigor                  | Type/Label: PERSON
Text: Caballo                   | Type/Label: PERSON
Text: Dolor                     | Type/Label: PERSON
Text: Hasta                    

In [17]:
# PROCESSING TEXT IN HYBRID or INTERSECTION (ESPAÑOL + LATIN)
# ================================================================
tInit = time.time()

# Extract person entities from Spanish model (label: "PER")
perEntitiesES = set([entity.text for entity in doc_ES.ents if "PER" == entity.label_])

# Extract person entities from Latin model (label: "PERSON")
perEntitiesLA = set([entity.text for entity in doc_LA.ents if "PERSON" ==entity.label_])

# Find confirmed person names detected by BOTH models (Intersection)
consensusPersons = perEntitiesES.intersection(perEntitiesLA)

# Display comparative intersection results
print(f"+ =================================================")
print(f"+ HYBRID or INTERSECTION (Spanish  &  Latin MODELS)")
print(f"+ Confirmed proper names (Detected by BOTH models):")
print(f"+ -------------------------------------------------")
LineBreak(1)

for name in consensusPersons:
  print(f"Confirmed Name: {name}")

LineBreak(1)
print(f"+ Summary metrics:")
print(f"+ -------------------------------------------------")
print(f"Spanish model candidate count : {len(perEntitiesES)}")
print(f"Latin model candidate count   : {len(perEntitiesLA)}")
print(f"Consensus candidates count    : {len(consensusPersons)}")

tEnd = time.time()
LineBreak(1)
print(f"+ Time to evaluate intersection: {tEnd - tInit:.4f} seconds")
print(f"+ =================================================")


+ =================================================
+ HYBRID or INTERSECTION (Spanish  &  Latin MODELS)
+ Confirmed proper names (Detected by BOTH models):
+ -------------------------------------------------
+ 
Confirmed Name: Dolor
Confirmed Name: Lucas
Confirmed Name: Caballo
Confirmed Name: Rodrigor
+ 
+ Summary metrics:
+ -------------------------------------------------
Spanish model candidate count : 7
Latin model candidate count   : 6
Consensus candidates count    : 4
+ 
+ Time to evaluate intersection: 0.0016 seconds
+ =================================================
